# 04 — Valuation Baseline Models

## Plate Value Intelligence

This notebook establishes transparent baseline models for predicting DVLA auction hammer prices from engineered licence-plate features.

### Objectives

- Load and validate the engineered feature table
- Preserve the event-based train, validation, and test split
- Separate identifiers, targets, governance fields, and model predictors
- Build simple benchmark models
- Evaluate predictions in both log-price and GBP terms
- Establish a performance baseline for later tree-based models

### Modelling principle

The final test event will remain untouched during model selection. Premium labels derived from hammer price will not be used as valuation predictors.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

In [2]:
PROJECT_ROOT = Path.cwd()
FEATURE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "plate_features_2025.csv"
)

print(f"Feature file: {FEATURE_FILE}")
print(f"File exists: {FEATURE_FILE.exists()}")


Feature file: /Users/osmanorka/Plate-Value-Intelligence/data/processed/plate_features_2025.csv
File exists: True


In [3]:
df = pd.read_csv(FEATURE_FILE)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")


Rows: 17,782
Columns: 75


In [4]:
validation_group_map = {
    "B270": "train",
    "B271": "train",
    "B272": "train",
    "B273": "train",
    "B274": "train",
    "B275": "train",
    "B276": "train",
    "B277": "validation",
    "B278": "test",
}

df["model_split"] = (
    df["event_code"]
    .map(validation_group_map)
)

assert df["model_split"].notna().all(), "Some events were not assigned to a model split."

df["model_split"].drop_duplicates().tolist()


Out[0]: ['train', 'validation', 'test']


In [5]:
split_summary = (
    df.groupby(
        ["model_split", "event_code"]
    )
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
    )
    .reset_index()
    .sort_values(["model_split", "event_code"])
)

split_summary


Out[0]: 
  model_split event_code  plate_count  median_price
0        test       B278         1981      1,500.00
1       train       B270         1966      1,485.00
2       train       B271         1969      1,510.00
3       train       B272         1983      1,510.00
4       train       B273         1983      1,510.00
5       train       B274         1981      1,440.00
6       train       B275         1970      1,510.00
7       train       B276         1972      1,370.00
8  validation       B277         1977      1,520.00
